# ComfyUI no Colab — instalação sob demanda por workflow

O *código* do ComfyUI fica no disco local do Colab (`/content`, rápido).
Modelos, outputs, inputs, workflows e o cache de custom nodes ficam no Drive.

**Só o ComfyUI-Manager é instalado sempre.** Todo o resto depende de quais
workflows você marcar na Célula 3 — o notebook lê o JSON de cada workflow,
descobre os `class_type` usados e instala apenas os pacotes necessários.

Ordem: **1 → 2 → 3 → 4 → (5 opcional) → 6**


In [ ]:
#@title 1. Montar Drive + instalar ComfyUI (local) { display-mode: "form" }
import os, subprocess, pathlib, json
from google.colab import drive

DRIVE_ROOT = '/content/drive'
DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'  #@param {type:"string"}
COMFY      = '/content/ComfyUI'

if not os.path.ismount(DRIVE_ROOT):
    drive.mount(DRIVE_ROOT)

def sh(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

if not os.path.exists(COMFY):
    sh(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}')
else:
    sh('git pull', cwd=COMFY, check=False)

MODEL_DIRS = ['checkpoints','loras','vae','clip','clip_vision','controlnet',
              'upscale_models','embeddings','unet','diffusion_models','ipadapter',
              'animatediff_models','animatediff_motion_lora','sams','style_models',
              'text_encoders','skintoken','trellis2','birefnet']
for d in MODEL_DIRS + ['ultralytics/bbox','ultralytics/segm']:
    pathlib.Path(f'{DRIVE_DATA}/models/{d}').mkdir(parents=True, exist_ok=True)
for d in ['output','input','user','workflows','node_cache']:
    pathlib.Path(f'{DRIVE_DATA}/{d}').mkdir(parents=True, exist_ok=True)

y = 'drive:\n  base_path: ' + DRIVE_DATA + '/models/\n'
y += ''.join(f'  {d}: {d}\n' for d in MODEL_DIRS)
y += '  ultralytics_bbox: ultralytics/bbox\n  ultralytics_segm: ultralytics/segm\n'
open(f'{COMFY}/extra_model_paths.yaml','w').write(y)

sh('pip install -q -r requirements.txt', cwd=COMFY)

# Manager: sempre. E o cache de nodes do Drive volta para o disco local.
CN = f'{COMFY}/custom_nodes'
CACHE = f'{DRIVE_DATA}/node_cache'
if not os.path.exists(f'{CN}/ComfyUI-Manager'):
    sh(f'git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager "{CN}/ComfyUI-Manager"')
    sh(f'pip install -q -r "{CN}/ComfyUI-Manager/requirements.txt"', check=False)

print('\n✅ Célula 1 OK. Coloque seus workflows .json em:', f'{DRIVE_DATA}/workflows')


In [ ]:
#@title 2. Registry de nodes (mapa class_type -> repositorio) { display-mode: "form" }
#@markdown Baixa o registry + os workflows direto do seu repo no GitHub.
import json, os, glob, subprocess, urllib.request

DRIVE_DATA = '/content/drive/MyDrive/ComfyUI_Data'
REPO = 'https://github.com/BloomRX/ComfyUI_Colab'
BRANCH = 'arena/01a05a82-comfyui-collab'
CKOUT = '/content/ComfyUI_Colab'

if os.path.exists(CKOUT):
    subprocess.run('git pull', shell=True, cwd=CKOUT, check=False)
else:
    subprocess.run(f'git clone --depth 1 -b {BRANCH} {REPO} {CKOUT}', shell=True, check=True)

LOCAL = f'{DRIVE_DATA}/node_registry.json'
REGISTRY = json.load(open(LOCAL if os.path.exists(LOCAL) else f'{CKOUT}/config/node_registry.json'))

PACKS      = REGISTRY['packs']
CLASS_MAP  = REGISTRY['class_map']
NATIVE_IGNORE = set(REGISTRY.get('native_ignore', []))
WF_DIRS = [f'{CKOUT}/Workflows', f'{DRIVE_DATA}/workflows']
print(f'OK: {len(PACKS)} pacotes, {len(CLASS_MAP)} nodes mapeados.')
for d in WF_DIRS:
    n = len(glob.glob(f'{d}/**/*.json', recursive=True)) if os.path.isdir(d) else -1
    print(('  %-45s %s' % (d, 'NAO EXISTE' if n < 0 else f'{n} workflow(s)')))


In [ ]:
#@title 3. Listar workflows e ver o que cada um precisa { display-mode: "form" }
#@markdown Roda e olha a lista numerada. A escolha e feita na Celula 4.
import json, os, glob, re, urllib.request

uuidpat = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-')
AUTO_RESOLVE = True  #@param {type:"boolean"}

files = []
for d in WF_DIRS:
    files += glob.glob(f'{d}/**/*.json', recursive=True)
files = sorted(set(files))
print('Procurado em:', WF_DIRS)
print('Encontrados :', len(files), 'arquivo(s)\n')

def classes_of(path):
    try: d = json.load(open(path, encoding='utf-8'))
    except Exception as e:
        print('  erro lendo', path, e); return set()
    out = set()
    if isinstance(d, dict) and 'nodes' in d:
        for n in d['nodes']:
            if n.get('type'): out.add(n['type'])
    elif isinstance(d, dict):
        for n in d.values():
            if isinstance(n, dict) and n.get('class_type'): out.add(n['class_type'])
    return out

EXTMAP = {}
if AUTO_RESOLVE:
    CANDS = [
      '/content/ComfyUI/custom_nodes/ComfyUI-Manager/extension-node-map.json',
      'https://raw.githubusercontent.com/Comfy-Org/ComfyUI-Manager/main/extension-node-map.json',
      'https://github.com/Comfy-Org/ComfyUI-Manager/raw/refs/heads/main/extension-node-map.json',
    ]
    for src in CANDS:
        try:
            if src.startswith('http'):
                req = urllib.request.Request(src, headers={'User-Agent': 'Mozilla/5.0'})
                raw = json.loads(urllib.request.urlopen(req, timeout=60).read())
            else:
                if not os.path.exists(src): continue
                raw = json.load(open(src, encoding='utf-8'))
            for repo, entry in raw.items():
                for cls in entry[0]:
                    EXTMAP.setdefault(cls, repo)
            print(f'Auto-resolve: {len(EXTMAP)} nodes conhecidos.\n'); break
        except Exception as e:
            print(' auto-resolve falhou:', str(e)[:60])

def resolve(cls):
    if cls in CLASS_MAP:
        p = CLASS_MAP[cls]; return (p, PACKS.get(p), 'registry')
    if cls in EXTMAP:
        url = EXTMAP[cls].rstrip('/')
        if url.endswith('.git'): url = url[:-4]
        return (url.rsplit('/', 1)[-1], url, 'auto')
    return None

WF = []   # lista global usada pela Celula 4
for f in files:
    cls = classes_of(f)
    need, auto, unk = {}, [], []
    for c in cls:
        if c in NATIVE_IGNORE or uuidpat.match(c): continue
        r = resolve(c)
        if r is None: unk.append(c)
        else:
            need[r[0]] = r[1]
            if r[2] == 'auto': auto.append(r[0])
    WF.append({'path': f, 'name': os.path.basename(f), 'need': need, 'unk': unk})

if not WF:
    print('!! Nenhum .json encontrado. Confira se a Celula 2 clonou o repo.')
else:
    print('=' * 70)
    for i, w in enumerate(WF, 1):
        print(f"[{i}] {w['name']}")
        print(f"    pacotes: {', '.join(sorted(w['need'])) or 'so nodes nativos'}")
        if w['unk']: print(f"    [!] sem fonte: {', '.join(sorted(w['unk'])[:5])}")
    print('=' * 70)
    print('\nAgora va na Celula 4 e escreva os numeros que quer. Ex: 1  ou  1,3')


In [ ]:
#@title 4. Instalar os custom nodes dos workflows escolhidos { display-mode: "form" }
#@markdown Numeros da lista da Celula 3, separados por virgula. `all` = todos.
SELECAO = "1"  #@param {type:"string"}

import os, subprocess

COMFY='/content/ComfyUI'; CN=f'{COMFY}/custom_nodes'
DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'
EXTRAS = REGISTRY.get('pack_extras', {})

sel = SELECAO.strip().lower()
if sel in ('all', 'todos', '*'):
    chosen = list(WF)
else:
    idx = [int(x) for x in re.findall(r'\d+', sel)]
    chosen = [WF[i-1] for i in idx if 1 <= i <= len(WF)]

if not chosen:
    raise SystemExit('Nada selecionado. Escreva um numero valido, ex: 1')

need = {}
for w in chosen: need.update(w['need'])
unk = sorted({u for w in chosen for u in w['unk']})

print('Workflows:', [w['name'] for w in chosen])
print('Pacotes  :', sorted(need) or '(nenhum)')
if unk: print('SEM FONTE (use o Manager depois):', unk)
print()

def sh(c, **k): print(f'$ {c}'); return subprocess.run(c, shell=True, check=False, **k)

apt = {a for pack in need for a in EXTRAS.get(pack, {}).get('apt', [])}
if apt:
    sh('apt-get -qq update && apt-get -qq install -y ' + ' '.join(sorted(apt)))

for pack, url in sorted(need.items()):
    if not url:
        print(f'!! {pack} sem URL — instale pelo Manager.'); continue
    dst = f'{CN}/{pack}'
    if os.path.exists(dst + '.disabled') and not os.path.exists(dst):
        os.rename(dst + '.disabled', dst); print(f'reativado {pack}')
    if not os.path.exists(dst):
        sh(f'git clone --depth 1 {url} "{dst}"')
    if not os.path.exists(dst):
        print(f'!! falha ao clonar {pack}'); continue
    if os.path.exists(f'{dst}/requirements.txt'):
        sh(f'pip install -q -r "{dst}/requirements.txt"')
    if os.path.exists(f'{dst}/install.py'):
        print(f'--- {pack}: install.py (compila CUDA, pode demorar MUITO)')
        sh('python install.py', cwd=dst)
    for m in EXTRAS.get(pack, {}).get('hf_models', []):
        out = f'{DRIVE_DATA}/models/{m["dest"]}'
        os.makedirs(out, exist_ok=True)
        for fn in m['files']:
            if not os.path.exists(f'{out}/{fn}'):
                sh(f'wget -q -c "{m["repo_url"]}/resolve/main/{fn}" -O "{out}/{fn}"')
    if EXTRAS.get(pack, {}).get('note'):
        print('   nota:', EXTRAS[pack]['note'])

for d in sorted(os.listdir(CN)):
    pth = f'{CN}/{d}'
    if not os.path.isdir(pth) or d == '__pycache__': continue
    if d == 'ComfyUI-Manager' or d in need: continue
    os.rename(pth, pth + '.disabled'); print(f'desativado {d}')

print('\nAtivos:', [d for d in sorted(os.listdir(CN))
                    if os.path.isdir(f'{CN}/{d}') and not d.endswith('.disabled')])


In [ ]:
#@title 5. (Opcional) Baixar modelo para o Drive { display-mode: "form" }
URL = ''  #@param {type:"string"}
PASTA = 'checkpoints'  #@param ["checkpoints","loras","vae","controlnet","upscale_models","unet","ipadapter","animatediff_models","clip_vision","sams","ultralytics/bbox"]
HF_TOKEN = ''  #@param {type:"string"}
import subprocess
DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'
if URL:
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}" ' if HF_TOKEN else ''
    subprocess.run(f'wget -c {hdr}--content-disposition "{URL}" -P "{DRIVE_DATA}/models/{PASTA}"',
                   shell=True, check=False)
else:
    print('Cole uma URL e rode de novo.')


In [ ]:
#@title 6. Ligar o ComfyUI { display-mode: "form" }
#@markdown `auto` = deixa o ComfyUI decidir (padrao). Use `lowvram` so se der OOM.
TUNEL = 'cloudflared'  #@param ["cloudflared","ngrok"]
VRAM  = 'auto'         #@param ["auto","highvram","lowvram","novram"]

import subprocess, threading, re, time, os, shlex

COMFY='/content/ComfyUI'; DRIVE_DATA='/content/drive/MyDrive/ComfyUI_Data'; PORT=8188

# ---------- Manager novo (pip) ----------
# Nas versoes recentes o Manager virou pacote pip do core; sem isso
# o --enable-manager so imprime warning e a UI nao aparece.
mreq = f'{COMFY}/manager_requirements.txt'
if os.path.exists(mreq):
    print('Instalando dependencias do Manager (pip)...')
    subprocess.run(f'pip install -q -r "{mreq}"', shell=True, check=False)

# ---------- tunel ----------
LINK = {'url': None}
if TUNEL == 'cloudflared':
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('wget -q -O /usr/local/bin/cloudflared '
          'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
          '&& chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    def tunnel():
        p = subprocess.Popen(['cloudflared','tunnel','--url',f'http://127.0.0.1:{PORT}'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
            if m and not LINK['url']:
                LINK['url'] = m.group(0)
                print('\n' + '='*62)
                print('  LINK DE ACESSO:', LINK['url'])
                print('  Espere aparecer "To see the GUI go to" antes de abrir.')
                print('='*62 + '\n')
    threading.Thread(target=tunnel, daemon=True).start()
    for _ in range(30):
        if LINK['url']: break
        time.sleep(1)
else:
    import getpass
    subprocess.run('pip install -q pyngrok', shell=True, check=True)
    from pyngrok import ngrok
    ngrok.kill(); ngrok.set_auth_token(getpass.getpass('Ngrok authtoken: '))
    print('\nLINK DE ACESSO:', ngrok.connect(PORT,'http').public_url, '\n')

# ---------- flags ----------
# --listen 0.0.0.0 e obrigatorio: com 127.0.0.1 o ComfyUI recusa o Host header
# do tunel (protecao anti DNS-rebinding) e o browser recebe 403.
args = ['--listen','0.0.0.0','--port',str(PORT),
        '--enable-cors-header','*',
        '--output-directory', f'{DRIVE_DATA}/output',
        '--input-directory',  f'{DRIVE_DATA}/input',
        '--user-directory',   f'{DRIVE_DATA}/user',
        '--preview-method','auto','--disable-auto-launch']
if VRAM != 'auto':
    args.append(f'--{VRAM}')

try:
    helptxt = subprocess.run(['python','main.py','--help'], cwd=COMFY,
                             capture_output=True, text=True, timeout=180).stdout
    args = [a for a in args if not (a.startswith('--') and a not in helptxt)]
    if '--enable-manager' in helptxt:
        args.append('--enable-manager')
except Exception as e:
    print('(nao consegui checar --help:', e, ')')

print('$ python main.py ' + ' '.join(shlex.quote(a) for a in args) + '\n')
!cd {COMFY} && python main.py {' '.join(shlex.quote(a) for a in args)}


## Sobre VRAM e abrir tudo junto

Os 6 GB são o pico de **um** workflow rodando. O problema de abrir os três juntos não é
o pico — é que o ComfyUI mantém em VRAM o último modelo carregado de cada execução, e
aba aberta com workflow grande também custa RAM de CPU. Misturar SDXL + AnimateDiff +
3D na mesma sessão faz o Colab começar a fazer swap e, no T4 (16 GB), OOM.

Por isso a Célula 3: uma sessão = um propósito.

- **Um workflow por sessão** é o ideal.
- Precisa alternar? Use **Free model and node cache** no menu do ComfyUI antes de trocar.
- `lowvram` na Célula 6 se estourar; `highvram` só em A100.

## Como funciona a seleção

1. Salve os workflows em `ComfyUI_Data/workflows/` (pode usar subpastas: `splash/`, `3d/`, `anim/`).
2. Célula 3 lê o JSON de cada um, extrai os `class_type` e cruza com o `class_map` do registry.
3. Célula 4 clona só os pacotes daqueles workflows, e renomeia o resto para `.disabled`
   — não apaga nada, e reativar é só marcar o workflow de novo.
4. Os repos ficam em cache em `ComfyUI_Data/node_cache/`, então a segunda sessão não baixa nada.

## Quando aparecer um node desconhecido

Se um workflow usa um node que não está no `class_map`, ele não é instalado e o ComfyUI
mostra "missing node". Aí: **Manager → Install Missing Custom Nodes**, veja o nome do pacote,
e adicione o par `"NomeDoClassType": "NomeDoPacote"` em `config/node_registry.json`
(mais o repo em `packs`, se for novo). Na próxima sessão ele entra sozinho.

**Me manda os três workflows** que eu preencho o registry com os nodes reais deles e testo o parser.
